<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 자기지도 학습 · 멀티모달 · 03. BYOL: negative 없이 bootstrap

## Bootstrap Your Own Latent: A New Approach to Self-Supervised Learning

- **원 논문:** [Bootstrap Your Own Latent: A New Approach to Self-Supervised Learning](https://arxiv.org/abs/2006.07733)
- **저자 / 발표:** Jean-Bastien Grill et al. · NeurIPS (2020)
- **난이도 / 예상 시간:** 중급 · 약 70분
- **선수 지식:** EMA, stop-gradient, normalized regression
- **로컬 학습 데이터:** `data/field_curriculum/multimodal_pairs.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 비대칭 online predictor와 stop-gradient target EMA로 두 view 표현을 회귀합니다.

**축소하거나 생략한 부분:** ImageNet·ResNet 대신 96개 8×8 이미지와 MLP를 사용하며 최종 성능 재현은 범위 밖입니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1, Figure 2 | online encoder/projector/predictor와 target | 비대칭 구조 |
| Eq. (1) | target parameter EMA | 수치 update |
| Eq. (2) | normalized prediction regression | 2−2 cosine 동치 |
| Eq. (3) | symmetric loss·target stop-gradient | loss·feature std |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** 두 view [B,C,H,W] → online/target [B,D] → prediction [B,D]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from copy import deepcopy
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(17)
np.random.seed(17)
torch.manual_seed(17)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(17)

DATA_PATH = Path("data/field_curriculum/multimodal_pairs.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
images = torch.tensor(raw["images"], dtype=torch.float32)
tokens = torch.tensor(raw["tokens"], dtype=torch.long)
class_ids = torch.tensor(raw["class_ids"], dtype=torch.long)
sequences = torch.tensor(raw["sequences"], dtype=torch.float32)
train_idx = torch.tensor(raw["train_idx"], dtype=torch.long)
test_idx = torch.tensor(raw["test_idx"], dtype=torch.long)
assert len(images) == len(tokens) == len(class_ids) == len(sequences)
assert images.shape[1:] == (1, 8, 8) and tokens.ndim == 2 and sequences.ndim == 3
print(
    f"pairs={len(images)}, classes={class_ids.unique().numel()}, "
    f"image={tuple(images.shape[1:])}, tokens={tokens.shape[1]}"
)
print(ACCELERATOR.summary())
print("dataset/index preprocessing stays on CPU, while neural training uses DEVICE")

## 포트폴리오 해설: 표현, 목적함수, 업데이트

$$\mathcal L_{\theta,\xi}=\left\|\bar q_\theta(z_\theta)-\operatorname{sg}(\bar z_\xi')
\right\|_2^2,\quad\xi\leftarrow\tau\xi+(1-\tau)\theta$$

- **기호 정의:** θ는 online, ξ는 target, q는 predictor, sg는 stop-gradient입니다.
- **수식의 역할:** negative 없이 비대칭 predictor와 느린 EMA target을 bootstrap 대상으로 씁니다.
- **구현 이유:** representation 붕괴·modality 누설·잘못된 positive 연결을
  코드에서 확인할 수 있도록 핵심 수식을 view 준비와 loss 경계로 분리했습니다.
- **Task/코드 대응:** Task P의 `prepare_views`, 논문 전용 `nn.Module`의
  `forward`, `compute_loss`, `training_step`이 §3.1, Figure 2을 구현합니다.
- **입출력 shape:** 두 view [B, 1, 8, 8] → online prediction/target [B, D]
- **평가:** normalized regression loss, target gradient 없음, representation 표준편차
- **원문 대비 한계:** ImageNet·ResNet 대신 96개 8×8 이미지와 MLP를 사용하며 최종 성능 재현은 범위 밖입니다.

구현은 `로컬 데이터 통계 → view/modality 준비 → model → objective/update → 평가`
순서입니다. 이 순서는 데이터 누설과 stop-gradient 경계를 코드에서 찾기 쉽게 합니다.

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §3.1, Figure 2
- **이 셀의 책임:** online encoder/projector/predictor와 target
- **Tensor shape 계약:** 두 view [B,C,H,W] → online/target [B,D] → prediction [B,D]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 비대칭 구조. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def byol_view(x):
    scale = 0.8 + 0.4 * torch.rand(len(x), 1, 1, 1, device=x.device)
    return (x * scale + 0.035 * torch.randn_like(x)).clamp(0, 1)


def backbone():
    return nn.Sequential(nn.Flatten(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 16))

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** Eq. (1)
- **이 셀의 책임:** target parameter EMA
- **Tensor shape 계약:** 두 view [B,C,H,W] → online/target [B,D] → prediction [B,D]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 수치 update. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
@torch.no_grad()
def ema_update(online: nn.Module, target: nn.Module, tau: float):
    for o, t in zip(online.parameters(), target.parameters()):
        t.data.mul_(tau).add_(o.data, alpha=1 - tau)


online = backbone()
target = deepcopy(online)
old = next(target.parameters()).clone()
next(online.parameters()).data.add_(0.2)
ema_update(online, target, 0.9)
assert torch.allclose(
    next(target.parameters()), 0.9 * old + 0.1 * next(online.parameters()), atol=1e-6
)

### 구현 단계 4 · 목적함수·학습 update

- **원문 위치:** Eq. (2)
- **이 셀의 책임:** normalized prediction regression
- **Tensor shape 계약:** 두 view [B,C,H,W] → online/target [B,D] → prediction [B,D]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 2−2 cosine 동치. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def normalized_mse(pred, target):
    return 2 - 2 * (F.normalize(pred, dim=1) * F.normalize(target, dim=1)).sum(1).mean()


predictor = nn.Sequential(nn.Linear(16, 24), nn.ReLU(), nn.Linear(24, 16))
x = images[:16]
v1, v2 = byol_view(x), byol_view(x)
test_loss = normalized_mse(predictor(online(v1)), target(v2).detach())
assert 0 <= float(test_loss) <= 4

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** §3.1, Figure 2 / Eq. (1) / Eq. (2) / Eq. (3)
- **이 셀의 책임:** online encoder/projector/predictor와 target / target parameter EMA / normalized prediction regression / symmetric loss·target stop-gradient
- **Tensor shape 계약:** 두 view [B,C,H,W] → online/target [B,D] → prediction [B,D]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 비대칭 구조 / 수치 update / 2−2 cosine 동치 / loss·feature std. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
online, target, predictor = ACCELERATOR.move(online, target, predictor)
optimizer = torch.optim.Adam([*online.parameters(), *predictor.parameters()], lr=0.012)
batch = ACCELERATOR.move(images[train_idx[:96]])
losses = []
stds = []
for _ in range(120):
    v1, v2 = byol_view(batch), byol_view(batch)
    p1, p2 = predictor(online(v1)), predictor(online(v2))
    with torch.no_grad():
        t1, t2 = target(v1), target(v2)
    loss = normalized_mse(p1, t2) + normalized_mse(p2, t1)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    ema_update(online, target, 0.97)
    losses.append(float(loss.detach().cpu()))
    stds.append(float(F.normalize(online(batch), dim=1).std(0).mean().detach().cpu()))
assert (
    losses[-1] < losses[0]
    and stds[-1] > 1e-3
    and all(p.grad is None for p in target.parameters())
)
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].plot(losses)
ax[0].set_title("BYOL loss")
ax[1].plot(stds)
ax[1].set_title("feature std (collapse check)")
plt.tight_layout()
plt.show()

### Task P1 · modality/view 준비와 핵심 모델

image·text·sequence의 shape를 먼저 고정하고 augmentation 또는 modality
경계를 명시한 뒤 논문 전용 `nn.Module`을 구성합니다.

### 구현 단계 6 · 핵심 연산·모델

- **원문 위치:** §3.1, Figure 2 / Eq. (1) / Eq. (2) / Eq. (3)
- **이 셀의 책임:** online encoder/projector/predictor와 target / target parameter EMA / normalized prediction regression / symmetric loss·target stop-gradient
- **Tensor shape 계약:** 두 view [B,C,H,W] → online/target [B,D] → prediction [B,D]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 비대칭 구조 / 수치 update / 2−2 cosine 동치 / loss·feature std. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    hidden_dim: int = 16
    momentum: float = 0.9
    learning_rate: float = 0.02
    steps: int = 14


def prepare_views(batch: Tensor) -> tuple[Tensor, Tensor]:
    first = (batch + 0.03 * torch.randn_like(batch)).clamp(0.0, 1.0)
    second = torch.flip(batch, dims=[3])
    return first, second


class ByolPortfolioModel(nn.Module):
    def __init__(self, config: PortfolioConfig):
        super().__init__()
        self.config = config
        self.online = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, config.hidden_dim),
        )
        self.target = deepcopy(self.online)
        for parameter in self.target.parameters():
            parameter.requires_grad_(False)
        self.target.eval()
        self.predictor = nn.Linear(config.hidden_dim, config.hidden_dim)

    def forward(
        self,
        first: Tensor,
        second: Tensor,
    ) -> tuple[Tensor, Tensor, Tensor, Tensor]:
        online_first = F.normalize(self.online(first), dim=1)
        online_second = F.normalize(self.online(second), dim=1)
        with torch.no_grad():
            target_first = F.normalize(self.target(first), dim=1)
            target_second = F.normalize(self.target(second), dim=1)
        assert not target_first.requires_grad
        assert not target_second.requires_grad
        return online_first, online_second, target_first, target_second

    def compute_loss(
        self,
        online_first: Tensor,
        online_second: Tensor,
        target_first: Tensor,
        target_second: Tensor,
    ) -> Tensor:
        prediction_first = F.normalize(self.predictor(online_first), dim=1)
        prediction_second = F.normalize(self.predictor(online_second), dim=1)
        first_to_second = 2.0 - 2.0 * (prediction_first * target_second.detach()).sum(
            dim=1
        )
        second_to_first = 2.0 - 2.0 * (prediction_second * target_first.detach()).sum(
            dim=1
        )
        return 0.5 * (first_to_second.mean() + second_to_first.mean())

    def update_target(self) -> None:
        with torch.no_grad():
            for target, online in zip(
                self.target.parameters(),
                self.online.parameters(),
                strict=True,
            ):
                target.mul_(self.config.momentum)
                target.add_(online, alpha=1.0 - self.config.momentum)

    def training_step(self, batch: Tensor) -> Tensor:
        first, second = prepare_views(batch)
        representations = self(first, second)
        return self.compute_loss(*representations)

### Task P2 · objective, stop-gradient, update

positive/negative, online/target 또는 masked/visible 경계를 확인하면서
`fit_portfolio_model`의 gradient와 EMA update 순서를 추적합니다.

### 구현 단계 7 · 목적함수·학습 update

- **원문 위치:** Eq. (1) / Eq. (3)
- **이 셀의 책임:** target parameter EMA / symmetric loss·target stop-gradient
- **Tensor shape 계약:** 두 view [B,C,H,W] → online/target [B,D] → prediction [B,D]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 수치 update / loss·feature std. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(model: ByolPortfolioModel, batch: Tensor) -> list[float]:
    trainable = [
        parameter for parameter in model.parameters() if parameter.requires_grad
    ]
    optimizer = torch.optim.Adam(trainable, lr=model.config.learning_rate)
    history = []
    for _ in range(model.config.steps):
        target_before = [
            parameter.detach().clone() for parameter in model.target.parameters()
        ]
        loss = model.training_step(batch)
        optimizer.zero_grad()
        loss.backward()
        assert all(parameter.grad is None for parameter in model.target.parameters())
        optimizer.step()
        assert all(
            torch.equal(before, after)
            for before, after in zip(
                target_before,
                model.target.parameters(),
                strict=True,
            )
        )
        model.update_target()
        history.append(float(loss.detach().cpu()))
    return history

### Task P3 · 표현 품질 평가

학습 loss와 retrieval·reconstruction·conditioning 지표를 분리해 계산합니다.
작은 로컬 pair의 수치는 원 논문의 downstream benchmark와 직접 비교하지 않습니다.

### 구현 단계 8 · 평가·완료 증거

- **원문 위치:** Eq. (2) / Eq. (3)
- **이 셀의 책임:** normalized prediction regression / symmetric loss·target stop-gradient
- **Tensor shape 계약:** 두 view [B,C,H,W] → online/target [B,D] → prediction [B,D]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 2−2 cosine 동치 / loss·feature std. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    model: ByolPortfolioModel,
    batch: Tensor,
) -> dict[str, float]:
    with torch.no_grad():
        first, second = prepare_views(batch)
        representations = model(first, second)
        regression = model.compute_loss(*representations)
        online_first, online_second, _, _ = representations
        online_views = torch.cat([online_first, online_second], dim=0)
        agreement = 1.0 - regression / 2.0
        spread = online_views.std(dim=0).mean()
    return {
        "symmetric_view_agreement": float(agreement.cpu()),
        "symmetric_regression_loss": float(regression.cpu()),
        "online_feature_spread": float(spread.cpu()),
    }


portfolio_train_index = train_idx[:64]
portfolio_test_index = test_idx
assert not torch.isin(portfolio_train_index, portfolio_test_index).any()
portfolio_train_images, portfolio_test_images = ACCELERATOR.move(
    images[portfolio_train_index],
    images[portfolio_test_index],
)
portfolio_model = ACCELERATOR.move(ByolPortfolioModel(PortfolioConfig()))
portfolio_history = fit_portfolio_model(portfolio_model, portfolio_train_images)
portfolio_metrics = evaluate_portfolio_model(portfolio_model, portfolio_test_images)
assert np.isfinite(portfolio_history).all()
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> ImageNet·ResNet 대신 96개 8×8 이미지와 MLP를 사용하며 최종 성능 재현은 범위 밖입니다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.